# SE-ResNet50 + Bilinear Pooling — TILDA Texture

Architecture SOTA pour textures :
- **SE-ResNet50** backbone (plus profond, plus de capacité)
- **Multi-scale Bilinear Pooling** sur layer3+layer4 (statistiques du 2e ordre = co-occurrence de features)
- **MixUp + CutMix** alternés (régularisation forte)
- **SWA** (Stochastic Weight Averaging = ensemble gratuit dans un seul modèle)
- TTA 36 variants : grille 9 crops × 4 flips

Objectif : val acc > 0.88

## 1. Setup

In [14]:
from pathlib import Path
import random, time

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

ROOT           = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = ROOT / 'data'
OUTPUT_DIR     = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'

for p in [CHECKPOINT_DIR, SUBMISSION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Root  :', ROOT)
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Root  : /home/onyxia/work/tilda-texture-classification
Device: cuda
GPU   : NVIDIA RTXA6000-48Q
VRAM  : 51.3 GB


## 2. Paramètres

In [15]:
N_SPLITS = 5
START_FOLD_FULL  = 1
START_FOLD_PATCH = 5

NUM_CLASSES = 8
NUM_WORKERS = 0

FULL_IMG_SIZE = (384, 576)
FULL_RESIZE   = (432, 648)

PATCH_RESIZE  = (512, 768)
PATCH_SIZE    = 384

BATCH_SIZE_FULL  = 32
BATCH_SIZE_PATCH = 32

EPOCHS_FULL  = 200
EPOCHS_PATCH = 250

LR_FULL  = 0.05
LR_PATCH = 0.05
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-3
LABEL_SMOOTHING = 0.1
PATIENCE = 70

BILINEAR_REDUCTION = 64   # 64*65//2 = 2080 features/echelle, x2 = 4160 total

MIXUP_ALPHA  = 0.4
CUTMIX_ALPHA = 1.0
CUTMIX_PROB  = 0.5

SWA_START_RATIO = 0.65

print('Config OK')
R = BILINEAR_REDUCTION
print(f'Features par echelle : {R*(R+1)//2}  |  Total FC input : {2*R*(R+1)//2}')


Config OK
Features par echelle : 2080  |  Total FC input : 4160


## 3. Données

In [16]:
def read_train_csv(path):
    try:
        df = pd.read_csv(path, sep=';')
        if len(df.columns) == 1:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if 'id' not in df.columns or 'label' not in df.columns:
        raise ValueError(f'Colonnes attendues id,label. Trouvees: {df.columns.tolist()}')
    df['id'] = df['id'].astype(int)
    df['label'] = df['label'].astype(int)
    return df

def resolve_image_path(folder, image_id):
    stem = str(int(image_id))
    for ext in ['.tif', '.tiff', '.png', '.jpg']:
        p = folder / f'{stem}{ext}'
        if p.exists(): return p
    matches = sorted(folder.glob(f'{stem}.*'))
    if matches: return matches[0]
    raise FileNotFoundError(f'Image introuvable id={image_id}')

train_csv = DATA_DIR / 'train.csv'
train_dir = DATA_DIR / 'train'
test_dir  = DATA_DIR / 'test'

df = read_train_csv(train_csv)
df['path'] = df['id'].map(lambda x: resolve_image_path(train_dir, x))

test_paths = sorted(test_dir.glob('*.*'), key=lambda p: int(p.stem))
test_df = pd.DataFrame({'id': [int(p.stem) for p in test_paths], 'path': test_paths})

print(df.head())
print('\nDistribution:', df['label'].value_counts().sort_index().to_dict())
print(f'Train: {len(df)}  Test: {len(test_df)}')


   id  label                                               path
0   1      4  /home/onyxia/work/tilda-texture-classification...
1   2      6  /home/onyxia/work/tilda-texture-classification...
2   3      7  /home/onyxia/work/tilda-texture-classification...
3   4      6  /home/onyxia/work/tilda-texture-classification...
4   5      7  /home/onyxia/work/tilda-texture-classification...

Distribution: {0: 300, 1: 300, 2: 262, 3: 300, 4: 300, 5: 299, 6: 300, 7: 300}
Train: 2361  Test: 789


## 4. Transforms + Dataset

In [17]:
full_train_tfms = transforms.Compose([
    transforms.Resize(FULL_RESIZE),
    transforms.RandomCrop(FULL_IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.08, scale=(0.01, 0.04), ratio=(0.3, 3.3)),
])

full_eval_tfms = transforms.Compose([
    transforms.Resize(FULL_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

patch_train_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.RandomCrop((PATCH_SIZE, PATCH_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.05, scale=(0.01, 0.03), ratio=(0.3, 3.3)),
])

patch_eval_full_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


class TildaDataset(Dataset):
    def __init__(self, frame, transform=None, has_labels=True):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('L')
        if self.transform is not None:
            image = self.transform(image)
        label = int(row['label']) if self.has_labels else -1
        return image, label, int(row['id'])


def make_loader(frame, transform, batch_size, shuffle=False, has_labels=True):
    ds = TildaDataset(frame, transform=transform, has_labels=has_labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


## 5. MixUp / CutMix

In [18]:
def rand_bbox(h, w, lam):
    cut_h = int(h * np.sqrt(1.0 - lam))
    cut_w = int(w * np.sqrt(1.0 - lam))
    cx, cy = np.random.randint(w), np.random.randint(h)
    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, w)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, h)
    return x1, y1, x2, y2


def mixup_cutmix(images, labels):
    B, C, H, W = images.shape
    idx = torch.randperm(B, device=images.device)
    if random.random() < CUTMIX_PROB:
        lam = float(np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA))
        x1, y1, x2, y2 = rand_bbox(H, W, lam)
        images = images.clone()
        images[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
        lam = 1.0 - (x2 - x1) * (y2 - y1) / (H * W)
    else:
        lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
        images = lam * images + (1.0 - lam) * images[idx]
    return images, labels, labels[idx], lam


def train_one_epoch_mix(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        images_m, la, lb, lam = mixup_cutmix(images, labels)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images_m)
        loss = lam * criterion(logits, la) + (1.0 - lam) * criterion(logits, lb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == la).sum().item()
        n += labels.size(0)
    return correct / n, total_loss / n

print('MixUp/CutMix OK')


MixUp/CutMix OK


## 6. Architecture — SE-ResNet50 + Bilinear Pooling

Le **Bilinear Pool** remplace le GlobalAvgPool :
- `x` feature map B×R×H×W → outer product B×R×R → upper triangle vectorisé
- Captures les **co-occurrences** de features → optimal pour textures
- Sign-sqrt + L2 norm pour stabilité numérique

**Multi-scale** : layer3 (1024ch) + layer4 (2048ch) → chacun réduit à `R=64` → 2080 features/échelle → 4160 total → FC(4160, 8)

In [19]:
# ─── SE blocks ───────────────────────────────────────────────────────────────

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c = x.shape[:2]
        w = self.pool(x).view(b, c)
        return x * self.fc(w).view(b, c, 1, 1)


class SEBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, se_reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, 1, bias=False)
        self.bn3   = nn.BatchNorm2d(planes * 4)
        self.se    = SEBlock(planes * 4, reduction=se_reduction)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * 4:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * 4, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * 4),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.se(self.bn3(self.conv3(out)))
        return F.relu(out + self.shortcut(x), inplace=True)


# ─── Bilinear Pooling ─────────────────────────────────────────────────────────

class BilinearPool(nn.Module):
    def __init__(self, in_channels, reduction=64):
        super().__init__()
        R = reduction
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, R, 1, bias=False),
            nn.BatchNorm2d(R),
            nn.ReLU(inplace=True),
        )
        rows, cols = torch.triu_indices(R, R)
        self.register_buffer('triu_r', rows)
        self.register_buffer('triu_c', cols)
        self.out_features = R * (R + 1) // 2

    def forward(self, x):
        x = self.reduce(x)                                  # B x R x H x W
        B, R, H, W = x.shape
        x = x.view(B, R, H * W)                            # B x R x HW
        cov = torch.bmm(x, x.transpose(1, 2)) / (H * W)   # B x R x R
        feat = cov[:, self.triu_r, self.triu_c]            # B x R*(R+1)/2
        feat = torch.sign(feat) * torch.sqrt(torch.abs(feat) + 1e-10)
        return F.normalize(feat, dim=1)


# ─── SE-ResNet50 + Multi-scale Bilinear ──────────────────────────────────────

class SEResNet50Bilinear(nn.Module):
    def __init__(self, num_classes=8, in_channels=1, reduction=64):
        super().__init__()
        self._in = 64
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make(64,  3, stride=1)  # -> 256 ch
        self.layer2 = self._make(128, 4, stride=2)  # -> 512 ch
        self.layer3 = self._make(256, 6, stride=2)  # -> 1024 ch
        self.layer4 = self._make(512, 3, stride=2)  # -> 2048 ch

        self.bpool3 = BilinearPool(1024, reduction=reduction)
        self.bpool4 = BilinearPool(2048, reduction=reduction)

        feat_dim = 2 * reduction * (reduction + 1) // 2
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(feat_dim, num_classes)

        self.apply(self._init_w)
        nn.init.xavier_normal_(self.fc.weight, gain=0.01)
        nn.init.zeros_(self.fc.bias)

    def _make(self, planes, n, stride=1):
        layers = []
        for s in [stride] + [1] * (n - 1):
            layers.append(SEBottleneck(self._in, planes, stride=s))
            self._in = planes * 4
        return nn.Sequential(*layers)

    def _init_w(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x  = self.stem(x)
        x  = self.layer1(x)
        x  = self.layer2(x)
        x3 = self.layer3(x)
        x4 = self.layer4(x3)
        f3 = self.bpool3(x3)
        f4 = self.bpool4(x4)
        return self.fc(self.dropout(torch.cat([f3, f4], dim=1)))


def seresnet50_bilinear(num_classes=8):
    return SEResNet50Bilinear(num_classes=num_classes, in_channels=1,
                              reduction=BILINEAR_REDUCTION)


m = seresnet50_bilinear()
n_params = sum(p.numel() for p in m.parameters())
print(f'SE-ResNet50-Bilinear params: {n_params/1e6:.2f}M')
print(f'Feature dim into FC: {2 * BILINEAR_REDUCTION * (BILINEAR_REDUCTION+1) // 2}')
del m


SE-ResNet50-Bilinear params: 26.26M
Feature dim into FC: 4160


## 7. Eval + TTA Inference

**36 predictions** par image test : grille 3×3 (9 crops) × 4 flips (none, H, V, HV)

In [20]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    y_true, y_pred = [], []
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        n += labels.size(0)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
    return correct / n, total_loss / n, np.array(y_true), np.array(y_pred)


def crop_grid(images, crop_size=384):
    _, _, h, w = images.shape
    top  = [0, max((h - crop_size) // 2, 0), max(h - crop_size, 0)]
    left = [0, max((w - crop_size) // 2, 0), max(w - crop_size, 0)]
    return [images[:, :, t:t+crop_size, l:l+crop_size]
            for t in top for l in left]


@torch.no_grad()
def predict_patch_tta36(model, loader, crop_size=384):
    model.eval()
    all_probs, all_ids = [], []
    for images, _, image_ids in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        prob_list = []
        for crop in crop_grid(images, crop_size=crop_size):
            for v in [crop,
                      torch.flip(crop, dims=[-1]),
                      torch.flip(crop, dims=[-2]),
                      torch.flip(crop, dims=[-2, -1])]:
                prob_list.append(torch.softmax(model(v), dim=1))
        probs = torch.stack(prob_list).mean(0)   # 36 variants averaged
        all_probs.append(probs.cpu())
        all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs).numpy(), np.array(all_ids)


@torch.no_grad()
def predict_full_tta(model, loader):
    model.eval()
    all_probs, all_ids = [], []
    for images, _, image_ids in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        variants = [images,
                    torch.flip(images, dims=[-1]),
                    torch.flip(images, dims=[-2]),
                    torch.flip(images, dims=[-2, -1])]
        probs = torch.stack([torch.softmax(model(v), dim=1) for v in variants]).mean(0)
        all_probs.append(probs.cpu())
        all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs).numpy(), np.array(all_ids)


def save_submission(ids, probs, path):
    sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1).astype(int)}).sort_values('id')
    sub.to_csv(path, index=False)
    print('Saved:', path)
    print(sub['label'].value_counts().sort_index())
    return sub

print('Inference OK')


Inference OK


## 8. Entraînement avec SWA

- **Phase normale** : SGD + Nesterov + CosineAnnealingLR
- **SWA** (dès epoch `SWA_START_RATIO × epochs`) : moyenne glissante des paramètres apprenables
- Après entraînement : recalcul des stats BN sur le modèle moyenné → évaluation SWA vs meilleur checkpoint

In [21]:
def _update_bn_manual(model, loader):
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.reset_running_stats()
    with torch.no_grad():
        for images, _, _ in tqdm(loader, desc='  BN update', leave=False):
            model(images.to(DEVICE))
    model.eval()


def fit_fold(model, train_loader, val_loader, run_name, epochs, lr):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM,
                                weight_decay=WEIGHT_DECAY, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.01)

    swa_start  = max(int(SWA_START_RATIO * epochs), 30)
    swa_params = None
    swa_n      = 0

    best_acc   = -1.0
    best_epoch = 0
    best_path  = CHECKPOINT_DIR / f'best_{run_name}.pt'
    history    = []
    t0         = time.time()

    for epoch in range(1, epochs + 1):
        train_acc, train_loss = train_one_epoch_mix(model, train_loader, criterion, optimizer)
        val_acc, val_loss, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        if epoch >= swa_start:
            swa_n += 1
            if swa_params is None:
                swa_params = {k: v.detach().clone() for k, v in model.named_parameters()}
            else:
                for k, v in model.named_parameters():
                    swa_params[k] = swa_params[k] + (v.detach() - swa_params[k]) / swa_n

        if val_acc > best_acc:
            best_acc, best_epoch = val_acc, epoch
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'best_acc': best_acc}, best_path)

        history.append({'epoch': epoch, 'train_acc': train_acc, 'train_loss': train_loss,
                        'val_acc': val_acc, 'val_loss': val_loss,
                        'lr': scheduler.get_last_lr()[0]})

        print(f'{run_name} | ep {epoch:03d} | train {train_acc:.4f}/{train_loss:.4f} | '
              f'val {val_acc:.4f}/{val_loss:.4f} | best {best_acc:.4f} @ {best_epoch}')

        if epoch - best_epoch >= PATIENCE:
            print(f'Early stop: no improvement for {PATIENCE} epochs')
            break

    # ── Finalise SWA ────────────────────────────────────────────────────────
    if swa_params is not None and swa_n >= 5:
        state = model.state_dict()
        for k, v in swa_params.items():
            state[k] = v
        model.load_state_dict(state)
        _update_bn_manual(model, train_loader)
        swa_val_acc, _, _, _ = evaluate(model, val_loader, criterion)
        print(f'SWA ({swa_n} snapshots) val acc: {swa_val_acc:.4f}  |  best ckpt: {best_acc:.4f}')
        if swa_val_acc >= best_acc:
            best_acc = swa_val_acc
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'best_acc': best_acc, 'is_swa': True}, best_path)
            print('-> SWA model saved as best')
        else:
            ckpt = torch.load(best_path, map_location=DEVICE, weights_only=True)
            model.load_state_dict(ckpt['model_state_dict'])
            print('-> Keeping best checkpoint')

    elapsed = (time.time() - t0) / 60
    return pd.DataFrame(history), best_path, best_acc, best_epoch, elapsed

print('fit_fold OK')


fit_fold OK


## 9. Boucle 5-fold générique

In [22]:
def run_5fold_experiment(experiment_name, model_factory,
                         train_tfms, val_tfms, test_tfms,
                         batch_size, epochs, lr,
                         start_fold=1, predict_kind='patch'):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_probs   = []
    ids_ref      = None
    fold_results = []

    test_loader = make_loader(test_df, test_tfms, batch_size=batch_size,
                              shuffle=False, has_labels=False)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['label']), start=1):
        fold_name  = f'{experiment_name}_fold{fold}'
        probs_path = CHECKPOINT_DIR / f'probs_{fold_name}.npy'
        ids_path   = CHECKPOINT_DIR / f'ids_{fold_name}.npy'

        if fold < start_fold:
            if probs_path.exists() and ids_path.exists():
                probs = np.load(probs_path)
                ids   = np.load(ids_path)
                fold_probs.append(probs)
                ids_ref = ids if ids_ref is None else ids_ref
                print(f'Fold {fold}: loaded saved probs {probs.shape}')
            else:
                print(f'Fold {fold}: skipped (no saved probs found)')
            continue

        print('\n' + '='*80)
        print(f'{experiment_name} | fold {fold}/{N_SPLITS}')
        print('='*80)

        tr_df = df.iloc[tr_idx].reset_index(drop=True)
        va_df = df.iloc[va_idx].reset_index(drop=True)
        train_loader = make_loader(tr_df, train_tfms, batch_size, shuffle=True)
        val_loader   = make_loader(va_df, val_tfms,   batch_size, shuffle=False)

        model = model_factory().to(DEVICE)
        hist_df, best_path, best_acc, best_epoch, elapsed = \
            fit_fold(model, train_loader, val_loader, fold_name, epochs, lr)
        hist_df.to_csv(OUTPUT_DIR / f'history_{fold_name}.csv', index=False)

        ckpt = torch.load(best_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(ckpt['model_state_dict'])

        if predict_kind == 'patch':
            probs, ids = predict_patch_tta36(model, test_loader, crop_size=PATCH_SIZE)
        else:
            probs, ids = predict_full_tta(model, test_loader)

        np.save(probs_path, probs)
        np.save(ids_path,   ids)
        if ids_ref is None:
            ids_ref = ids
        else:
            assert np.array_equal(ids_ref, ids), 'Test ids different entre folds!'

        fold_probs.append(probs)
        save_submission(ids, probs,
                        SUBMISSION_DIR / f'submission_{fold_name}_tta_labels0.csv')

        fold_results.append({
            'experiment': experiment_name, 'fold': fold,
            'best_val_acc': best_acc, 'best_epoch': best_epoch,
            'training_time_min': elapsed,
        })

        del model
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(fold_results)
    results_df.to_csv(OUTPUT_DIR / f'results_{experiment_name}.csv', index=False)

    if len(fold_probs) == N_SPLITS:
        mean_probs = np.mean(fold_probs, axis=0)
        np.save(CHECKPOINT_DIR / f'probs_{experiment_name}_5fold.npy', mean_probs)
        np.save(CHECKPOINT_DIR / f'ids_{experiment_name}_5fold.npy',   ids_ref)
        save_submission(ids_ref, mean_probs,
                        SUBMISSION_DIR / f'submission_{experiment_name}_5fold_tta_labels0.csv')
    else:
        print(f'Pas de CSV 5-fold final : {len(fold_probs)}/{N_SPLITS} folds OK')

    display(results_df)
    if not results_df.empty:
        print(f"Mean val acc: {results_df['best_val_acc'].mean():.4f} "
              f"+/- {results_df['best_val_acc'].std():.4f}")
    return results_df

print('run_5fold_experiment OK')


run_5fold_experiment OK


## 10. Run A — Patch 384×384 (run principal)

Patch model = meilleur pour TILDA (0.8372 fold1 avec SE-ResNet18).
Avec SE-ResNet50 + Bilinear + MixUp/CutMix + SWA + 250 epochs → objectif > 0.88

In [23]:
patch_results = run_5fold_experiment(
    experiment_name = 'seresnet50_bilinear_patch_v1',
    model_factory   = seresnet50_bilinear,
    train_tfms      = patch_train_tfms,
    val_tfms        = patch_eval_full_tfms,
    test_tfms       = patch_eval_full_tfms,
    batch_size      = BATCH_SIZE_PATCH,
    epochs          = EPOCHS_PATCH,
    lr              = LR_PATCH,
    start_fold      = START_FOLD_PATCH,
    predict_kind    = 'patch',
)


Fold 1: loaded saved probs (789, 8)
Fold 2: loaded saved probs (789, 8)
Fold 3: loaded saved probs (789, 8)
Fold 4: loaded saved probs (789, 8)

seresnet50_bilinear_patch_v1 | fold 5/5


  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

seresnet50_bilinear_patch_v1_fold5 | ep 001 | train 0.1196/2.0856 | val 0.1292/2.0802 | best 0.1292 @ 1


  0%|          | 0/60 [00:00<?, ?it/s]

...

SWA (89 snapshots) val acc: 0.1123  |  best ckpt: 0.8792
-> Keeping best checkpoint


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_seresnet50_bilinear_patch_v1_fold5_tta_labels0.csv
label
0    128
1     85
2     88
3    106
4     92
5     94
6    103
7     93
Name: count, dtype: int64
Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_seresnet50_bilinear_patch_v1_5fold_tta_labels0.csv
label
0    131
1     97
2    105
3     86
4     87
5     96
6     97
7     90
Name: count, dtype: int64


,experiment,fold,best_val_acc,best_epoch,training_time_min
0,seresnet50_bilinear_patch_v1,5,0.879237,245,95.0305


Mean val acc: 0.8792 +/- nan


## 11. Run B — Full image (optionnel)

A lancer APRES Run A. Peut améliorer l'ensemble si val acc > 0.82.

In [24]:
# Décommenter pour lancer Run B full-image
# full_results = run_5fold_experiment(
#     experiment_name = 'seresnet50_bilinear_full_v1',
#     model_factory   = seresnet50_bilinear,
#     train_tfms      = full_train_tfms,
#     val_tfms        = full_eval_tfms,
#     test_tfms       = full_eval_tfms,
#     batch_size      = BATCH_SIZE_FULL,
#     epochs          = EPOCHS_FULL,
#     lr              = LR_FULL,
#     start_fold      = START_FOLD_FULL,
#     predict_kind    = 'full',
# )
print('Run B commenté. Décommenter si voulu.')


Run B commenté. Décommenter si voulu.


## 12. Ensemble

Si Run B (full) a été lancé et a val acc > 0.82, faire 0.4×full + 0.6×patch.
Sinon, utiliser patch seul (déjà le 5-fold moyen).

In [25]:
patch_probs_path = CHECKPOINT_DIR / 'probs_seresnet50_bilinear_patch_v1_5fold.npy'
patch_ids_path   = CHECKPOINT_DIR / 'ids_seresnet50_bilinear_patch_v1_5fold.npy'
full_probs_path  = CHECKPOINT_DIR / 'probs_seresnet50_bilinear_full_v1_5fold.npy'

if not patch_probs_path.exists():
    print('Patch 5-fold probs introuvable. Lancer Run A.')
else:
    patch_probs = np.load(patch_probs_path)
    patch_ids   = np.load(patch_ids_path)

    if full_probs_path.exists():
        full_probs = np.load(full_probs_path)
        ensemble_probs = 0.40 * full_probs + 0.60 * patch_probs
        ens_name = 'seresnet50_bilinear_full_patch_v1_ensemble'
        print('Ensemble full (0.4) + patch (0.6)')
    else:
        ensemble_probs = patch_probs
        ens_name = 'seresnet50_bilinear_patch_v1_only'
        print('Patch seul (Run B non disponible)')

    np.save(CHECKPOINT_DIR / f'probs_{ens_name}.npy', ensemble_probs)
    save_submission(patch_ids, ensemble_probs,
                    SUBMISSION_DIR / f'submission_{ens_name}_labels0.csv')


Patch seul (Run B non disponible)
Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_seresnet50_bilinear_patch_v1_only_labels0.csv
label
0    131
1     97
2    105
3     86
4     87
5     96
6     97
7     90
Name: count, dtype: int64
